# Nik1 — distil a teacher into a 3 MB student (Kaggle GPU)

Labels, not parameters, are the bottleneck for a task-specialist dataset. This
notebook labels tens of thousands of developer utterances with a real instruct
model, then trains a small Nik1 student that runs on device in milliseconds.

Two paths, both in `nik1/python/torch/distill.py`:

| teacher | what it gives you | cost |
|---|---|---|
| `oracle` | the deterministic template generator — exactly the behaviour the runtime must match | free, unlimited |
| `hf` | a real LLM (Qwen/Llama…) covering phrasings the templates never had | GPU minutes |

Every teacher label is passed through the **same grammar repair + schema
validation** the runtime uses, so the student never learns to emit an invalid
call.

In [ ]:
import os, subprocess
REPO = "https://github.com/NikitHamal/Pixelforge.git"
BRANCH = "feat/nik1-on-device-models"
if not os.path.exists("/kaggle/working/Pixelforge"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/kaggle/working/Pixelforge"], check=True)
%cd /kaggle/working/Pixelforge
!pip -q install transformers accelerate
!python3 -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"

In [ ]:
# Oracle distillation: the cheapest way to a strong student. ~5 minutes.
!cd nik1/python && python3 torch/distill.py --teacher oracle \
    --n 20000 --steps 4000 --d-model 192 --layers 5 --heads 6 --d-ff 512 \
    --out /kaggle/working/nik1-distilled

In [ ]:
# Optional: a real LLM teacher for phrasings the templates do not cover.
# Pick a small instruct model that fits a T4 (0.5B-1.5B works well).
!cd nik1/python && python3 torch/distill.py --teacher hf \
    --model Qwen/Qwen2.5-0.5B-Instruct --n 6000 --steps 3000 \
    --d-model 192 --layers 5 --heads 6 --d-ff 512 \
    --out /kaggle/working/nik1-distilled-hf

In [ ]:
import json, os
for OUT in ["/kaggle/working/nik1-distilled", "/kaggle/working/nik1-distilled-hf"]:
    if not os.path.isdir(OUT): continue
    print("=" * 60); print(OUT)
    for f in sorted(os.listdir(OUT)):
        print(f"  {f:40} {os.path.getsize(os.path.join(OUT,f))/1024:8.1f} KB")
    m = [f for f in os.listdir(OUT) if f.endswith(".metrics.json")]
    if m: print(json.dumps(json.load(open(os.path.join(OUT, m[0]))), indent=1))

In [ ]:
# Evaluate in the runtime, then download.
import shutil, os
from IPython.display import FileLink, display
MODELS = "/kaggle/working/Pixelforge/nik1/js/models"
for f in ["nik1-route-distilled.nik1", "nik1-route-distilled.json"]:
    src = f"/kaggle/working/nik1-distilled/{f}"
    if os.path.exists(src):
        shutil.copy(src, os.path.join(MODELS, "nik1-route.nik1" if f.endswith(".nik1") else "nik1-route.json"))
!cd /kaggle/working/Pixelforge && node nik1/tests/test_runtime.js --bench
for f in ["nik1-route-distilled.nik1", "nik1-route-distilled.json"]:
    p = f"/kaggle/working/nik1-distilled/{f}"
    if os.path.exists(p): display(FileLink(p))